# 14. Energy Analysis

Create an OpenStudio energy model from one TopologicPy room and inspect its spaces. Model preparation is kept separate from a full annual simulation.

**Goal:** Complete one small workflow. Run the cells from top to bottom in a fresh kernel.

**Extra dependencies:** `openstudio==3.11.0`. No simulation engine is needed for the cells here.

All examples use metres unless stated otherwise.


In [ ]:
import os
os.environ["TOPOLOGICPY_CORE_BACKEND"] = "pythonocc"

import topologicpy
from topologicpy.Core import Core

assert topologicpy.__version__ == "0.9.86", "Use TopologicPy 0.9.86 for these tutorials."
assert type(Core.Backend()).__name__ == "PythonOCCBackend", "Restart with the pythonocc backend."
print("TopologicPy", topologicpy.__version__, "—", type(Core.Backend()).__name__)


## Make the analytical room

The tutorial uses a supplied office template and London weather/design-day files. Creating a model is quick; running an annual EnergyPlus simulation requires the OpenStudio command-line tools and is beyond this short introduction.


In [ ]:
from pathlib import Path
from topologicpy.Cell import Cell
from topologicpy.EnergyModel import EnergyModel

room = Cell.Prism(width=4, length=3, height=3, placement="bottom")
data = Path("data") / "energy"
model = EnergyModel.ByTopology(
    room,
    osModelPath=str(data / "office.osm"),
    weatherFilePath=str(data / "london.epw"),
    designDayFilePath=str(data / "london.ddy"),
    floorLevels=[0, 3],
    glazingRatio=0.2,
    buildingName="Tutorial Room"
)
assert model is not None


## Inspect and save

The space count and total floor area verify the conversion. The OSM file can be opened in OpenStudio. It contains template assumptions that should be reviewed before any real analysis.


In [ ]:
spaces = model.getSpaces()
floor_area = 0
for space in spaces:
    floor_area = floor_area + space.floorArea()
print("Spaces:", len(spaces), "Floor area:", floor_area)
assert len(spaces) == 1
assert abs(floor_area - 12) < 0.001
output = Path("results")
output.mkdir(exist_ok=True)
assert EnergyModel.ExportToOSM(model, str(output / "14_room.osm"), overwrite=True)


## Try it yourself

Change the room width and confirm the resulting floor area. Later, explore `EnergyModel.Run` with a configured OpenStudio CLI.
